<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> 책의 보조 코드, 저자: <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>코드 저장소: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# PyTorch 버퍼(Buffers) 이해하기

본질적으로 PyTorch 버퍼는 매개변수와 유사하게 PyTorch 모듈이나 모델과 연관된 텐서 속성이지만, 매개변수와 달리 버퍼는 학습 중에 업데이트되지 않습니다.

PyTorch의 버퍼는 GPU 연산을 다룰 때 특히 유용한데, 모델의 매개변수와 함께 디바이스 간(예: CPU에서 GPU로) 전송되어야 하기 때문입니다. 매개변수와 달리 버퍼는 그래디언트 계산이 필요하지 않지만, 모든 연산이 올바르게 수행되도록 올바른 디바이스에 있어야 합니다.

3장에서는 `self.register_buffer`를 통해 PyTorch 버퍼를 사용하는데, 이는 책에서 간략하게만 설명됩니다. 개념과 목적이 즉시 명확하지 않으므로, 이 코드 노트북은 실습 예제와 함께 더 긴 설명을 제공합니다.

## 버퍼 없는 예제

3장의 코드를 기반으로 한 다음 코드를 가정해 보겠습니다. 이 버전은 버퍼를 제외하도록 수정되었습니다. LLM에서 사용되는 인과적 셀프 어텐션 메커니즘(causal self-attention mechanism)을 구현합니다:

In [ ]:
import torch
import torch.nn as nn

class CausalAttentionWithoutBuffers(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

다음과 같이 몇 가지 예제 데이터에서 모듈을 초기화하고 실행할 수 있습니다:

In [ ]:
torch.manual_seed(123)

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

batch = torch.stack((inputs, inputs), dim=0)
context_length = batch.shape[1]
d_in = inputs.shape[1]
d_out = 2

ca_without_buffer = CausalAttentionWithoutBuffers(d_in, d_out, context_length, 0.0)

with torch.no_grad():
    context_vecs = ca_without_buffer(batch)

print(context_vecs)

지금까지 모든 것이 잘 작동했습니다.

하지만 LLM을 학습할 때는 일반적으로 GPU를 사용하여 프로세스를 가속화합니다. 따라서 `CausalAttentionWithoutBuffers` 모듈을 GPU 디바이스로 전송해보겠습니다.

이 작업은 GPU가 장착된 환경에서 코드를 실행해야 한다는 점에 유의하세요.

In [ ]:
print("머신에 GPU 있음:", torch.cuda.is_available())

batch = batch.to("cuda")
ca_without_buffer.to("cuda");

이제 코드를 다시 실행해보겠습니다:

In [ ]:
with torch.no_grad():
    context_vecs = ca_without_buffer(batch)

print(context_vecs)

코드를 실행한 결과 오류가 발생했습니다. 무엇이 일어났나요? GPU의 텐서와 CPU의 텐서 사이의 행렬 곱셈을 시도한 것 같습니다. 하지만 우리는 모듈을 GPU로 이동했는데요!?


일부 텐서의 디바이스 위치를 다시 확인해보겠습니다:

In [ ]:
print("W_query.device:", ca_without_buffer.W_query.weight.device)
print("mask.device:", ca_without_buffer.mask.device)

In [ ]:
type(ca_without_buffer.mask)

보시다시피 `mask`는 GPU로 이동되지 않았습니다. 그 이유는 가중치(예: `W_query.weight`)와 같은 PyTorch 매개변수가 아니기 때문입니다.

즉, `.to("cuda")`를 통해 수동으로 GPU로 이동해야 합니다:

In [ ]:
ca_without_buffer.mask = ca_without_buffer.mask.to("cuda")
print("mask.device:", ca_without_buffer.mask.device)

코드를 다시 시도해보겠습니다:

In [ ]:
with torch.no_grad():
    context_vecs = ca_without_buffer(batch)

print(context_vecs)

이번에는 작동했습니다!

하지만 개별 텐서를 GPU로 이동하는 것을 기억하는 것은 번거로울 수 있습니다. 다음 섹션에서 볼 수 있듯이, `mask`를 버퍼로 등록하기 위해 `register_buffer`를 사용하는 것이 더 쉽습니다.

## 버퍼를 사용한 예제

이제 인과적 어텐션 클래스를 수정하여 인과적 `mask`를 버퍼로 등록해보겠습니다:

In [ ]:
import torch
import torch.nn as nn

class CausalAttentionWithBuffer(nn.Module):

    def __init__(self, d_in, d_out, context_length,
                 dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # 기존:
        # self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

        # 새로운 방법:
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

이제 편리하게도 모듈을 GPU로 이동하면 마스크도 GPU에 위치하게 됩니다:

In [ ]:
ca_with_buffer = CausalAttentionWithBuffer(d_in, d_out, context_length, 0.0)
ca_with_buffer.to("cuda")

print("W_query.device:", ca_with_buffer.W_query.weight.device)
print("mask.device:", ca_with_buffer.mask.device)

In [ ]:
with torch.no_grad():
    context_vecs = ca_with_buffer(batch)

print(context_vecs)

위에서 볼 수 있듯이 텐서를 버퍼로 등록하면 우리의 삶이 훨씬 쉬워집니다: GPU와 같은 대상 디바이스로 텐서를 수동으로 이동하는 것을 기억할 필요가 없습니다.

## 버퍼와 `state_dict`

- 일반 텐서에 비해 PyTorch 버퍼의 또 다른 장점은 모델의 `state_dict`에 포함된다는 것입니다
- 예를 들어, 버퍼가 없는 인과적 어텐션 객체의 `state_dict`를 살펴보겠습니다

In [ ]:
ca_without_buffer.state_dict()

- 위의 `state_dict`에는 마스크가 포함되지 않습니다
- 하지만 아래의 `state_dict`에는 마스크를 버퍼로 등록했기 때문에 마스크가 *포함*되어 있습니다

In [ ]:
ca_with_buffer.state_dict()

- `state_dict`는 예를 들어 학습된 PyTorch 모델을 저장하고 로드할 때 유용합니다
- 이 특정한 경우에서 `mask`를 저장하고 로드하는 것이 학습 중에 변경되지 않기 때문에 그다지 유용하지 않을 수 있습니다. 따라서 시연을 위해 모든 `1`이 `2`로 변경된 것처럼 수정되었다고 가정해보겠습니다:

In [ ]:
ca_with_buffer.mask[ca_with_buffer.mask == 1.] = 2.
ca_with_buffer.mask

- 그러면 모델을 저장하고 로드할 때 마스크가 수정된 값으로 복원되는 것을 볼 수 있습니다

In [ ]:
torch.save(ca_with_buffer.state_dict(), "model.pth")

new_ca_with_buffer = CausalAttentionWithBuffer(d_in, d_out, context_length, 0.0)
new_ca_with_buffer.load_state_dict(torch.load("model.pth"))

new_ca_with_buffer.mask

- 이는 버퍼를 사용하지 않는 경우에는 해당되지 않습니다:

In [ ]:
ca_without_buffer.mask[ca_without_buffer.mask == 1.] = 2.

torch.save(ca_without_buffer.state_dict(), "model.pth")

new_ca_without_buffer = CausalAttentionWithoutBuffers(d_in, d_out, context_length, 0.0)
new_ca_without_buffer.load_state_dict(torch.load("model.pth"))

new_ca_without_buffer.mask